In [2]:
# resume_fft_detector.py
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ds import (
    WatermarkOnTheFlyDataset,
    discover_dataset_files,
    make_train_image_augmentations,
    get_test_aug,
)
from attack import (
    make_clean_aug,
    make_jpeg_aug,
    make_msg_app_combo,
    make_down_up_attack,
    make_blur_aug,
    make_random_crop_attack,
    make_occlusion_block,
    make_geom_aug,
)
import time
from watermark import get_watermarking_mask, get_watermarking_pattern
import os
from utils.thr import get_best_thrs
import diffusers
from diffusers import DPMSolverMultistepScheduler
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from psnr import (
    eval_watermark_results,
    get_batch_results,
)

from sklearn import metrics
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------- Config (adjust if needed) -----------------

NAME = "tree_ring"
DATA_DIR = "./verifier_dataset_stablediff_ring"
CHECKPOINT_PATH = ""
EVAL_RESULT_SAVE_DIR = os.path.join("./eval_results/", NAME)
BATCH_SIZE = 8
NUM_WORKERS = 0
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1  # how often to save full checkpoint
IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = get_test_aug(IMAGE_SIZE)
INCLUDE_MASK_PATCH = False
INCLUDE_PSNR_L1 = True  # whether to include PSNR metric in dataset output

# Watermarking parameters (should match those used during watermark embedding)
W_MASK_SHAPE = "circle"
W_CHANNEL = 0
W_RADIUS = 10
W_STRENGTH = 0.99
W_PATTERN = "ring"
# -------------------------------------------------------------

# ---------------- reproducibility ----------------
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
# -------------------------------------------------

# ---------------- Load or define PIPE and TEXT_EMBEDDINGS ----------------
# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)

model_id = "stabilityai/stable-diffusion-2-1-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
scheduler = DPMSolverMultistepScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = InversableStableDiffusionPipeline.from_pretrained(
    model_id,
    scheduler=scheduler,
    torch_dtype=torch.float16,
    revision="fp16",
    verbose=False,
)
diffusers.utils.logging.disable_progress_bar()
pipe.set_progress_bar_config(disable=True)
pipe = pipe.to(device)

TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
PIPE = pipe  # make sure 'pipe' is in scope

# -----------------------------------------------------------------------

watermarking_mask = get_watermarking_mask(
    pipe.get_random_latents(),
    w_mask_shape=W_MASK_SHAPE,
    w_channel=W_CHANNEL,
    w_radius=W_RADIUS,
    device=device,
)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=SEED,
    w_pattern=W_PATTERN,
    w_radius=W_RADIUS,
    device=device,
    strength=W_STRENGTH,
    shape=None,
)


#  ---------------- Prepare datasets and dataloaders ----------------
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
print(val_labels[:50])
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=None,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr_l1=INCLUDE_PSNR_L1,
    watermarking_mask=watermarking_mask,
    gt_patch=gt_patch,
    psnr_return_prob=False,
)
# val_ds.file_paths = val_ds.file_paths[:50]
val_ds.image_aug_prob = 1  # always apply augmentations during validation
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
crit = nn.CrossEntropyLoss()

# ----------------------------------------------------------------------
# Evaluation loop (resume) with pretty printing and PSNR metrics
# ----------------------------------------------------------------------
testing_times = 5
crit = nn.CrossEntropyLoss()
total_start = time.time()


def avg(values):
    return sum(values) / len(values) if values else float("nan")


# Define attack transforms
attack_factories = {
    "clean": lambda: make_clean_aug(IMAGE_SIZE),
    "jpeg_strong": lambda: make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=60),
    "msg_app_combo": lambda: make_msg_app_combo(IMAGE_SIZE),
    "down_up": lambda: make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5),
    "blur": lambda: make_blur_aug(IMAGE_SIZE),
    "random_crop": lambda: make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)),
    "occlusion": lambda: make_occlusion_block(IMAGE_SIZE, box_frac=0.25),
    "geom_warp": lambda: make_geom_aug(IMAGE_SIZE),
    "train_aug_mix": lambda: make_train_image_augmentations(IMAGE_SIZE),
}

# Use existing val_loader
dataset = val_loader.dataset

os.makedirs(EVAL_RESULT_SAVE_DIR, exist_ok=True)

result_df = []

for attack_name, aug_builder in attack_factories.items():

    attack_result_dir = os.path.join(EVAL_RESULT_SAVE_DIR, attack_name)
    os.makedirs(attack_result_dir, exist_ok=True)

    dataset.image_aug = aug_builder()  # ← change augmentation in-place

    # metric trackers for this attack
    all_preds, all_gts = [], []

    for test_i in range(testing_times):

        val_loader.dataset.set_return_reversed_latents(True)
        preds, gts = eval_watermark_results(
            lambda x: get_batch_results(x, gt_patch, watermarking_mask),
            val_loader,
            device=DEVICE,
        )

        val_loader.dataset.set_return_reversed_latents(False)
        # Accumulate
        all_preds.extend(preds)
        all_gts.extend(gts)

    l1_fpr, l1_tpr, l1_thresholds = metrics.roc_curve(
        all_gts, [-p["l1_metric"] for p in all_preds], pos_label=1
    )
    best_l1_thr = get_best_thrs(l1_fpr, l1_tpr, l1_thresholds)
    psnr_fpr, psnr_tpr, psnr_thresholds = metrics.roc_curve(
        all_gts, [p["psnr_metric"] for p in all_preds], pos_label=1
    )
    best_psnr_thr = get_best_thrs(psnr_fpr, psnr_tpr, psnr_thresholds)
    l1_auc = metrics.auc(l1_fpr, l1_tpr)
    psnr_auc = metrics.auc(psnr_fpr, psnr_tpr)

    # Remember the thresholds for later attack if the attack is clean
    if attack_name == "clean":
        clean_best_l1_thr = best_l1_thr
        clean_best_psnr_thr = best_psnr_thr

    # Compute final accuracy at best thresholds
    l1_preds = [1 if -p["l1_metric"] >= clean_best_l1_thr else 0 for p in all_preds]
    psnr_preds = [
        1 if p["psnr_metric"] >= clean_best_psnr_thr else 0 for p in all_preds
    ]
    l1_acc = sum([1 if p == gt else 0 for p, gt in zip(l1_preds, all_gts)]) / len(
        all_gts
    )
    psnr_acc = sum([1 if p == gt else 0 for p, gt in zip(psnr_preds, all_gts)]) / len(
        all_gts
    )

    attack_eval_results = {
        "attack_name": attack_name,
        "preds": all_preds,
        "gts": all_gts,
        "psnrs" : [p["psnr_metric"] for p in all_preds],
        "l1s" : [-p["l1_metric"] for p in all_preds],
        "l1_acc": l1_acc,
        "l1_auc": l1_auc,
        "psnr_acc": psnr_acc,
        "psnr_auc": psnr_auc,
        "best_l1_thr": best_l1_thr,
        "best_psnr_thr": best_psnr_thr,
        "clean_best_l1_thr": clean_best_l1_thr,
        "clean_best_psnr_thr": clean_best_psnr_thr,
    }
    result_df.append({
        "attack": attack_name,
        "l1_acc": l1_acc,
        "l1_auc": l1_auc,
        "best_l1_thr": best_l1_thr,
        "psnr_acc": psnr_acc,
        "psnr_auc": psnr_auc,
        "best_psnr_thr": best_psnr_thr,
    })
    # Save results to file
    result_save_path = os.path.join(attack_result_dir, "eval_results.pt")
    torch.save(attack_eval_results, result_save_path)

    # Summary for this attack
    print("\n" + "-" * 80)
    print(f"AVERAGE ({attack_name}) over {testing_times} runs")
    print("-" * 80)
    print(f"{'Metric':<20} {'Avg Value':>18}")
    print("-" * 42)
    print(f"{'L1 Accuracy':<20} {l1_acc:>18.4f}")
    print(f"{'L1 AUROC':<20}   {l1_auc:>18.4f}")
    print(f"{'PSNR Accuracy':<20} {psnr_acc:>18.4f}")
    print(f"{'PSNR AUROC':<20} {psnr_auc:>18.4f}")
    print(f"{'Best L1 Thr':<20} {best_l1_thr:>18.4f}")
    print(f"{'Best PSNR Thr':<20} {best_psnr_thr:>18.4f}")
    # threshold used.
    print(
        f"Thresholds used for Acc: L1={clean_best_l1_thr:.4f}, PSNR={clean_best_psnr_thr:.4f}"
    )
    print("-" * 42)
    print("#" * 80)


print("\n" + "=" * 80)
print("All evaluations complete.")
print("=" * 80)

df_results = pd.DataFrame(result_df)
display(df_results)
# save results to CSV
csv_path = os.path.join(EVAL_RESULT_SAVE_DIR, "attack_eval_summary.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved summary results to {csv_path}")

Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-693130d6-5f5d1a856c806e4a6d3c760e;c4fb3875-d220-462d-8a5e-30c01d4c3ac8)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1-base/revision/fp16.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password..
Will try to load from local cache.
Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffus

(1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0)



--------------------------------------------------------------------------------
AVERAGE (clean) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.8867
L1 AUROC                           0.9241
PSNR Accuracy                    0.8067
PSNR AUROC                       0.8085
Best L1 Thr                    -73.3750
Best PSNR Thr                    6.4510
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (jpeg_strong) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.7147
L1 AUROC                           0.7893
PSNR Accuracy                    0.6120
PSNR AUROC                       0.6607
Best L1 Thr                    -74.1875
Best PSNR Thr                    6.3680
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (msg_app_combo) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.5533
L1 AUROC                           0.6879
PSNR Accuracy                    0.5347
PSNR AUROC                       0.5989
Best L1 Thr                    -77.1250
Best PSNR Thr                    6.1082
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################


...


--------------------------------------------------------------------------------
AVERAGE (occlusion) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.8520
L1 AUROC                           0.9071
PSNR Accuracy                    0.7600
PSNR AUROC                       0.7910
Best L1 Thr                    -74.1875
Best PSNR Thr                    6.3391
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (geom_warp) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.7493
L1 AUROC                           0.8342
PSNR Accuracy                    0.6613
PSNR AUROC                       0.7432
Best L1 Thr                    -74.0000
Best PSNR Thr                    6.3327
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (train_aug_mix) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.7027
L1 AUROC                           0.7758
PSNR Accuracy                    0.6240
PSNR AUROC                       0.6692
Best L1 Thr                    -74.1250
Best PSNR Thr                    6.3612
Thresholds used for Acc: L1=-73.3750, PSNR=6.4510
------------------------------------------
################################################################################

All evaluations complete.


,attack,l1_acc,l1_auc,best_l1_thr,psnr_acc,psnr_auc,best_psnr_thr
0,clean,0.886667,0.924140,-73.3750,0.806667,0.808522,6.451019
1,jpeg_strong,0.714667,0.789335,-74.1875,0.612000,0.660738,6.368002
2,msg_app_combo,0.553333,0.687877,-77.1250,0.534667,0.598859,6.108187
3,down_up,0.746667,0.835621,-75.5000,0.633333,0.711892,6.158803
4,blur,0.630667,0.712070,-75.6250,0.541333,0.611018,5.735331
5,random_crop,0.740000,0.853607,-75.6875,0.633333,0.749945,6.016031
6,occlusion,0.852000,0.907078,-74.1875,0.760000,0.791007,6.339147
7,geom_warp,0.749333,0.834206,-74.0000,0.661333,0.743220,6.332750
8,train_aug_mix,0.702667,0.775846,-74.1250,0.624000,0.669160,6.361215


Saved summary results to ./eval_results/tree_ring\attack_eval_summary.csv
